# card_stack

> Card stack UI operations — navigation, viewport, mode switching, and response builders

In [ ]:
#| default_exp routes.decomposition.card_stack

In [ ]:
#| export
from typing import List, Dict, Any, Tuple

from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_CARD_WIDTH
from cjm_fasthtml_card_stack.routes.handlers import (
    build_slots_response, build_nav_response,
    card_stack_navigate, card_stack_update_viewport, card_stack_save_width,
)

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.segment_card import (
    create_segment_card_renderer
)
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.core import (
    _to_segments, _load_decomp_context, _get_decomp_state,
    _build_card_stack_state, _update_decomp_state,
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Response Builders

Shared helpers that assemble OOB response tuples for card stack operations.

- **Slots only** — early returns and exit-split-mode (just viewport section swaps)
- **Navigation response** — navigation and enter-split-mode (slots + progress + focus)

In [ ]:
#| export
def _make_renderer(
    urls: DecompUrls,  # URL bundle
    is_split_mode: bool = False,  # Whether split mode is active
    caret_position: int = 0,  # Caret position for split mode
) -> Any:  # Card renderer callback
    """Create a segment card renderer with captured URLs and mode state."""
    return create_segment_card_renderer(
        split_url=urls.split,
        merge_url=urls.merge,
        enter_split_url=urls.enter_split,
        exit_split_url=urls.exit_split,
        is_split_mode=is_split_mode,
        caret_position=caret_position,
    )

In [ ]:
#| export
def _build_slots_oob(
    segment_dicts: List[Dict[str, Any]],  # Serialized segments
    state: CardStackState,  # Card stack viewport state
    urls: DecompUrls,  # URL bundle
    caret_position: int = 0,  # Caret position for split mode
) -> List[Any]:  # OOB slot elements
    """Build OOB slot updates for the viewport sections."""
    is_split = state.active_mode == "split"
    return build_slots_response(
        card_items=_to_segments(segment_dicts),
        state=state,
        config=DECOMP_CS_CONFIG,
        ids=DECOMP_CS_IDS,
        urls=urls.card_stack,
        render_card=_make_renderer(urls, is_split_mode=is_split, caret_position=caret_position),
    )

In [ ]:
#| export
def _build_nav_response(
    segment_dicts: List[Dict[str, Any]],  # Serialized segments
    state: CardStackState,  # Card stack viewport state
    urls: DecompUrls,  # URL bundle
    caret_position: int = 0,  # Caret position for split mode
) -> Tuple:  # OOB elements (slots + progress + focus)
    """Build OOB response for navigation and mode changes."""
    is_split = state.active_mode == "split"
    return build_nav_response(
        card_items=_to_segments(segment_dicts),
        state=state,
        config=DECOMP_CS_CONFIG,
        ids=DECOMP_CS_IDS,
        urls=urls.card_stack,
        render_card=_make_renderer(urls, is_split_mode=is_split, caret_position=caret_position),
        progress_label="Segment",
    )

## Navigation Handler

In [ ]:
#| export
def _handle_decomp_navigate(
    workflow: StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
    direction: str,  # Navigation direction: "up", "down", "first", "last", "page_up", "page_down"
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with progress and focus
    """Navigate to a different segment in the viewport using OOB slot swaps."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    segments = _to_segments(ctx.segment_dicts)
    
    state = _build_card_stack_state(ctx)
    renderer = _make_renderer(urls)
    
    result = card_stack_navigate(
        direction=direction,
        card_items=segments,
        state=state,
        config=DECOMP_CS_CONFIG,
        ids=DECOMP_CS_IDS,
        urls=urls.card_stack,
        render_card=renderer,
        progress_label="Segment",
    )
    
    _update_decomp_state(workflow, session_id, focused_index=state.focused_index)
    return result

## Enter/Exit Split Mode Handlers

In [ ]:
#| export
def _handle_decomp_enter_split_mode(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    segment_index: int,  # Index of segment to enter split mode for
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with split mode active for focused segment
    """Enter split mode for a specific segment."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    _update_decomp_state(workflow, session_id, focused_index=segment_index)

    state = _build_card_stack_state(ctx, active_mode="split")
    state.focused_index = segment_index
    return _build_nav_response(ctx.segment_dicts, state, urls, caret_position=0)

In [ ]:
#| export
def _handle_decomp_exit_split_mode(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with split mode deactivated
    """Exit split mode."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    state = _build_card_stack_state(ctx)
    return _build_slots_oob(ctx.segment_dicts, state, urls)

## Update Viewport Handler

Handler for updating the viewport when card count changes. Does a full viewport swap (outerHTML) since the number of slots changes.

In [ ]:
#| export
def _handle_decomp_update_viewport(
    workflow: StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
    visible_count: int,  # New number of visible cards
    urls: DecompUrls,  # URL bundle for decomposition routes
):  # Full viewport component (outerHTML swap)
    """Update the viewport with a new card count.

    Does a full viewport swap because the number of slots changes.
    Saves the new visible_count to state for subsequent operations.
    """
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    segments = _to_segments(ctx.segment_dicts)
    
    state = _build_card_stack_state(ctx)
    renderer = _make_renderer(urls)
    
    result = card_stack_update_viewport(
        visible_count=visible_count,
        card_items=segments,
        state=state,
        config=DECOMP_CS_CONFIG,
        ids=DECOMP_CS_IDS,
        urls=urls.card_stack,
        render_card=renderer,
    )
    
    _update_decomp_state(workflow, session_id, visible_count=state.visible_count)
    return result

## Save Width Handler

Saves the card stack width to server state for persistence across page loads.

In [ ]:
#| export
def _handle_decomp_save_width(
    workflow: StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
    card_width: int,  # Card stack width in rem
) -> None:  # No response body (swap=none on client)
    """Save the card stack width to server state.
    
    Called via debounced HTMX POST from the width slider.
    Returns nothing since the client uses hx-swap='none'.
    """
    session_id = get_session_id(sess)
    decomp_state = _get_decomp_state(workflow, session_id)
    current_width = decomp_state.get("card_width", DEFAULT_CARD_WIDTH)
    state = CardStackState(card_width=current_width)
    card_stack_save_width(state, card_width, DECOMP_CS_CONFIG)
    _update_decomp_state(workflow, session_id, card_width=state.card_width)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()